# Medical QA Bot - Qdrant Hybrid Search & Ingestion Notebook

This notebook demonstrates how to set up **Hybrid Search** with **Qdrant Vector Database** for the Medical QA dataset.

### Why Hybrid Search?
- **Dense Vector Search** (e.g., `PubMedBERT` embeddings): Captures conceptual meaning, intent, and semantic relationships, finding relevant passages even when exact words differ.
- **Sparse Vector Search** (e.g., `BM25` keyword scoring): Captures precise keyword matches, medical acronyms, specific condition names, and exact terminology.
- **Hybrid Search (Reciprocal Rank Fusion - RRF)**: Merges scores from both Dense and Sparse representations in Qdrant to return higher quality, highly relevant search results.

---

In [1]:
import os
import math
import re
from collections import Counter
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

load_dotenv()

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Hardware accelerator: {device}")

Hardware accelerator: mps


In [2]:
# Connect to Qdrant Database
QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
QDRANT_URL = os.getenv("QDRANT_URL", f"http://{QDRANT_HOST}:{QDRANT_PORT}")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", None)

print(f"Attempting connection to Qdrant at: {QDRANT_URL}...")

client = None
if QDRANT_URL:
    try:
        remote_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, check_compatibility=False, timeout=5)
        remote_client.get_collections()
        client = remote_client
        print(f"Successfully connected to Qdrant Vector Server at {QDRANT_URL}!")
    except Exception as e:
        print(f"Standalone Qdrant server not available at {QDRANT_URL} ({e}).")

if client is None:
    client = QdrantClient(":memory:")
    print("Connected to Qdrant In-Memory Session!")

Attempting connection to Qdrant at: https://67b23fca-0482-4a4e-9cdd-e579a9f6eced.europe-west3-0.gcp.cloud.qdrant.io...
Successfully connected to Qdrant Vector Server at https://67b23fca-0482-4a4e-9cdd-e579a9f6eced.europe-west3-0.gcp.cloud.qdrant.io!


In [3]:
# Load Medical Dataset
DATASET_PATH = "dataset/medquad.csv"
df = pd.read_csv(DATASET_PATH)

# Preprocess text columns
df["question"] = df["question"].fillna("").astype(str)
df["answer"] = df["answer"].fillna("").astype(str)
df["focus_area"] = df["focus_area"].fillna("General").astype(str)
df["source"] = df["source"].fillna("Unknown").astype(str)

df["combined_text"] = df["question"] + " " + df["answer"]

print(f"Loaded {len(df)} rows from {DATASET_PATH}")
display(df.head())

Loaded 16412 rows from dataset/medquad.csv


,question,answer,source,focus_area,combined_text
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,What is (are) Glaucoma ? Glaucoma is a group o...
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma,What causes Glaucoma ? Nearly 2.7 million peop...
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma,What are the symptoms of Glaucoma ? Symptoms o...
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma,What are the treatments for Glaucoma ? Althoug...
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,What is (are) Glaucoma ? Glaucoma is a group o...


In [4]:
# Initialize Dense Model (PubMedBERT)
DENSE_MODEL_NAME = "NeuML/pubmedbert-base-embeddings"
print(f"Loading Dense Embedding Model: {DENSE_MODEL_NAME}...")

try:
    dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device, local_files_only=True)
except Exception:
    dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

DENSE_VECTOR_SIZE = 768
print(f"Dense model initialized successfully. Embedding dimension: {DENSE_VECTOR_SIZE}")

Loading Dense Embedding Model: NeuML/pubmedbert-base-embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense model initialized successfully. Embedding dimension: 768


In [ ]:
# Initialize Official Qdrant FastEmbed BM25 Sparse Encoder
from fastembed import SparseTextEmbedding
from qdrant_client import models

print("Loading FastEmbed BM25 Sparse Vectorizer (Qdrant/bm25)...")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

def encode_sparse(text):
    embed = list(sparse_model.embed([str(text)]))[0]
    return models.SparseVector(
        indices=embed.indices.tolist(),
        values=embed.values.tolist()
    )

print("FastEmbed BM25 Sparse Encoder initialized successfully!")

In [6]:
# Create Hybrid Collection in Qdrant with Disk Storage
COLLECTION_NAME = "medical_knowledge_base_hybrid"

existing_collections = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME in existing_collections:
    print(f"Re-creating existing collection '{COLLECTION_NAME}'...")
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "text-dense": models.VectorParams(
            size=DENSE_VECTOR_SIZE,
            distance=models.Distance.COSINE,
            on_disk=True
        )
    },
    sparse_vectors_config={
        "text-sparse": models.SparseVectorParams(
            index=models.SparseIndexParams(on_disk=True)
        )
    }
)

print(f"Created hybrid collection '{COLLECTION_NAME}' with disk storage enabled (dense: {DENSE_VECTOR_SIZE}, sparse: BM25)!")

Created hybrid collection 'medical_knowledge_base_hybrid' with disk storage enabled (dense: 768, sparse: BM25)!


In [7]:
# Batch Embed & Ingest Data into Qdrant
BATCH_SIZE = 256
texts = df["combined_text"].tolist()

print(f"Generating dense embeddings for {len(texts)} documents...")
dense_vectors = dense_model.encode(
    texts,
    # batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Building Qdrant points and streaming batch insertions...")
points = []
for idx, row in df.iterrows():
    dense_vec = dense_vectors[idx].tolist()
    sparse_vec = encode_sparse(row["combined_text"])
    
    point = models.PointStruct(
        id=idx,
        vector={
            "text-dense": dense_vec,
            "text-sparse": sparse_vec
        },
        payload={
            "question": row["question"],
            "answer": row["answer"],
            "source": row["source"],
            "focus_area": row["focus_area"]
        }
    )
    points.append(point)

# Upload points in batches
UPLOAD_BATCH = 500
for i in tqdm(range(0, len(points), UPLOAD_BATCH), desc="Uploading to Qdrant"):
    batch = points[i : i + UPLOAD_BATCH]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)

print(f"\nSuccessfully ingested {len(points)} records into Qdrant!")

Generating dense embeddings for 16412 documents...


Batches:   0%|          | 0/513 [00:00<?, ?it/s]

Building Qdrant points and streaming batch insertions...


Uploading to Qdrant:   0%|          | 0/33 [00:00<?, ?it/s]


Successfully ingested 16412 records into Qdrant!


In [8]:
# Verify Collection Info in Qdrant
info = client.get_collection(COLLECTION_NAME)
print(f"Collection Name: {COLLECTION_NAME}")
print(f"Status: {info.status}")
print(f"Points Count: {info.points_count}")

Collection Name: medical_knowledge_base_hybrid
Status: yellow
Points Count: 16412


In [ ]:
# Search Functions: Dense, Sparse, and Hybrid RRF
def dense_search(query_text, top_k=5):
    query_vector = dense_model.encode(query_text, normalize_embeddings=True).tolist()
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using="text-dense",
        limit=top_k
    )
    return res.points

def sparse_search(query_text, top_k=5):
    sparse_vector = encode_sparse(query_text)
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=sparse_vector,
        using="text-sparse",
        limit=top_k
    )
    return res.points

def hybrid_search(query_text=None, query=None, top_k=5):
    q_text = query_text or query
    query_dense = dense_model.encode(q_text, normalize_embeddings=True).tolist()
    query_sparse = encode_sparse(q_text)
    
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=query_dense, using="text-dense", limit=top_k * 3),
            models.Prefetch(query=query_sparse, using="text-sparse", limit=top_k * 3),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    )
    return res.points

In [10]:
# Execute and Compare Search Strategies
QUERY = "What is the treatment of high blood pressure?"
print(f"Search Query: '{QUERY}'\n")

print("=== 1. Dense Vector Search (PubMedBERT) ===")
dense_results = dense_search(QUERY, top_k=3)
for r in dense_results:
    print(f"Score: {r.score:.4f} | Category: {r.payload['focus_area']}")
    print(f"Question: {r.payload['question']}")
    print(f"Answer: {r.payload['answer'][:150]}...\n")

print("=== 2. Sparse Vector Search (BM25) ===")
sparse_results = sparse_search(QUERY, top_k=3)
for r in sparse_results:
    print(f"Score: {r.score:.4f} | Category: {r.payload['focus_area']}")
    print(f"Question: {r.payload['question']}")
    print(f"Answer: {r.payload['answer'][:150]}...\n")

print("=== 3. Hybrid Search (Dense + Sparse with Reciprocal Rank Fusion) ===")
hybrid_results = hybrid_search(QUERY, top_k=3)
for r in hybrid_results:
    print(f"Score: {r.score:.4f} | Category: {r.payload['focus_area']}")
    print(f"Question: {r.payload['question']}")
    print(f"Answer: {r.payload['answer'][:150]}...\n")

Search Query: 'What is the treatment of high blood pressure?'

=== 1. Dense Vector Search (PubMedBERT) ===
Score: 0.7996 | Category: High Blood Pressure
Question: What are the treatments for High Blood Pressure ?
Answer: In most cases, the goal is probably to keep your blood pressure below 140/90 mmHg (130/80 if you have diabetes or chronic kidney disease). Normal bloo...

Score: 0.7982 | Category: High Blood Pressure
Question: What are the treatments for High Blood Pressure ?
Answer: High blood pressure is treated with lifestyle changes and medicines. Treatment can help control blood pressure, but it will not cure high blood pressu...

Score: 0.7578 | Category: Blood Pressure Medicines
Question: What is (are) Blood Pressure Medicines ?
Answer: High blood pressure, also called hypertension, usually has no symptoms. But it can cause serious problems such as stroke, heart failure, heart attack ...

=== 2. Sparse Vector Search (BM25) ===
Score: 66.4089 | Category: High Blood Pressure
Ques